#1. Install deps

In [8]:
import sys
import os

if 'google.colab' in sys.modules:
    if not os.path.exists('/content/NLP'):
        !git clone -b lab-12 https://github.com/AndrianaNahirna/NLP.git

    %cd /content/NLP

    !pip install -r requirements.txt -q
    !pip install -q transformers accelerate bitsandbytes
    !git fetch origin
    !git reset --hard origin/lab-12

    sys.path.append('/content/NLP')

print("Середовище налаштовано.")

/content/NLP
remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Total 5 (delta 4), reused 5 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 430 bytes | 430.00 KiB/s, done.
From https://github.com/AndrianaNahirna/NLP
   259236f..9b3f792  lab-12     -> origin/lab-12
HEAD is now at 9b3f792 update eval_agent.py
Середовище налаштовано.


#2. Data / test cases

In [2]:
TEST_CASES = [
    {"id": "case_01", "text": "Я оплатив замовлення №123456, але статус не змінився."}, # Простий кейс
    {"id": "case_02", "text": "У мене списало гроші двічі, поверніть!"}, # Missing data (немає ID замовлення)
    {"id": "case_03", "text": "Добрий день як там пароль змінити??"}, # Noisy text
    {"id": "case_04", "text": "Де мої речі?"}, # Порожній результат інструментів
    {"id": "case_05", "text": "Дякую, все запрацювало, ви найкращі!"}, # Зайвий виклик (немає проблеми)
    {"id": "case_06", "text": "Не можу зайти в акаунт, мій email test@mail.com"}, # Account issue
    {"id": "case_07", "text": "Додаток вилітає при спробі відкрити налаштування."}, # Technical issue
    {"id": "case_08", "text": "Скільки коштує доставка для замовлення 98765?"}, # Billing + order ID
    {"id": "case_09", "text": "Order 445566 is delayed, why?"}, # Інша мова/неоднозначність
    {"id": "case_10", "text": "Я забув логін. Допоможіть."} # Missing data для account
]

#3. Tool definitions

In [3]:
from sentiment.src.tools import classify_ticket, extract_entities, validate_required_fields

sample_text = "Я оплатив замовлення №998877, але гроші зняло двічі. Мій email test@mail.com"

print("Classification:", classify_ticket(sample_text))
print("Extraction:", extract_entities(sample_text))

Classification: {'category': 'billing', 'confidence': 0.9}
Extraction: {'emails': ['test@mail.com'], 'order_ids': ['998877']}


#4. Tool call logger

In [4]:
import os
from sentiment.src.tool_logger import log_tool_call, LOG_FILE

# Перевіряємо, чи існує папка для логів
os.makedirs(os.path.dirname(LOG_FILE), exist_ok=True)

# Очищуємо файл логів перед новим запуском тестів
with open(LOG_FILE, "w", encoding="utf-8") as f:
    pass

print(f"Логер готовий: {LOG_FILE}")

Логер готовий: ../docs/tool_logs_lab12.jsonl


#5. Agent design

Агент ініціалізується в src/agent.py. Ми використовуємо модель unsloth/llama-3-8b-Instruct-bnb-4bit. Модель завантажиться автоматично під час першого імпорту з sentiment.src.agent.

#6. Baseline LLM without tools

In [5]:
from sentiment.src.agent import run_baseline

sample_input = TEST_CASES[0]['text']
print(f"User: {sample_input}")
print(f"Baseline (Тільки LLM): {run_baseline(sample_input)}")

Завантаження моделі...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

User: Я оплатив замовлення №123456, але статус не змінився.
Baseline (Тільки LLM): Дякуємо за повідомлення! Ми перевіримо ситуацію з вашим замовленням №123456. Якщо статус не змінився, можливо, є технічні проблеми або помилка в обробці замовлення. Наші фахівці зв'яжуться з вами найближчим часом, щоб допомогти вирішити цю ситуацію.


#7. Agent with tools

In [6]:
from sentiment.src.agent import run_agent

# Тестуємо агента з tools на тому ж прикладі
print(f"User: {sample_input}")
print(f"Agent (Tools + LLM): {run_agent('test_run_01', sample_input)}")

User: Я оплатив замовлення №123456, але статус не змінився.
Agent (Tools + LLM): Дякуємо за повідомлення. Ми перевіримо статус вашого замовлення №123456. Якщо у вас виникла проблема з оплатою, будь ласка, повідомте нас детальніше про ситуацію. Ми будемо радий допомогти вам вирішити цю проблему.


#8. Run 10 test cases

In [14]:
from sentiment.src.eval_agent import run_evaluation

with open(LOG_FILE, "w", encoding="utf-8") as f:
    pass

# Ця функція прожене всі 10 кейсів через Baseline та Agent і запише логи
run_evaluation(TEST_CASES)

Початок тестування

Task ID: case_01
User Input: Я оплатив замовлення №123456, але статус не змінився.

[BASELINE - Без Tools]:
Дякуємо за повідомлення! Ми перевіримо ситуацію з вашим замовленням №123456. Будь ласка, надайте нам більше інформації про замовлення, наприклад, дату оплати та очікуване виконання. Ми постараємося вирішити проблему якнайшвидше.

[AGENT - З Tools]:
Дякуємо за повідомлення! Ми збираємося допомогти вам з вашим замовленням №123456. Побачимо, що відбувається з його статусом. Будь ласка, дайте нам кілька хвилин, щоб перевірити ситуацію.
--------------------------------------------------

Task ID: case_02
User Input: У мене списало гроші двічі, поверніть!

[BASELINE - Без Tools]:
Дякуємо за повідомлення про проблеми з грошовими операціями. Ми співпрацюємо з нашими фінансовими партнерами, щоб вирішити цю ситуацію. Будь ласка, надайте нам більше інформації про ваш випадок, наприклад, дату та суму операції, щоб ми могли допомогти вам якнайшвидше.

[AGENT - З Tools]:
Дя

#9. Tool call logs

In [11]:
import json

# Виводимо останні кілька записів з логів
print("Останні збережені логи:")
with open(LOG_FILE, "r", encoding="utf-8") as f:
    logs = [json.loads(line) for line in f]

for log in logs:
    print(json.dumps(log, indent=2, ensure_ascii=False))

Останні збережені логи:
{
  "timestamp": "2026-05-17T09:50:43.210511+00:00",
  "task_id": "test_run_01",
  "tool_name": "classify_ticket",
  "input": {
    "text": "Я оплатив замовлення №123456, але статус не змінився."
  },
  "output": {
    "category": "general_inquiry",
    "confidence": 0.5
  },
  "success": true,
  "error": null
}
{
  "timestamp": "2026-05-17T09:50:43.211685+00:00",
  "task_id": "test_run_01",
  "tool_name": "extract_entities",
  "input": {
    "text": "Я оплатив замовлення №123456, але статус не змінився."
  },
  "output": {
    "emails": null,
    "order_ids": [
      "123456"
    ]
  },
  "success": true,
  "error": null
}
{
  "timestamp": "2026-05-17T09:50:43.211868+00:00",
  "task_id": "test_run_01",
  "tool_name": "validate_required_fields",
  "input": {
    "emails": null,
    "order_ids": [
      "123456"
    ]
  },
  "output": {
    "is_valid": true,
    "missing_fields": []
  },
  "success": true,
  "error": null
}
{
  "timestamp": "2026-05-17T09:52:50.8

#10. Metrics

In [15]:
import json
from collections import defaultdict

# Завантажуємо логи
with open("../docs/tool_logs_lab12.jsonl", "r", encoding="utf-8") as f:
    logs = [json.loads(line) for line in f]

# Автоматичні метрики
total_calls = len(logs)
success_calls = sum(1 for log in logs if log.get("success"))
error_calls = total_calls - success_calls

# Tool call success rate
success_rate = (success_calls / total_calls * 100) if total_calls > 0 else 0
# Tool error rate
error_rate = (error_calls / total_calls * 100) if total_calls > 0 else 0

# Average tool calls per task
total_tasks = len(TEST_CASES)
calls_per_task = total_calls / total_tasks

# % задач, де agent використав обидва (>= 2) tools
logs_by_task = defaultdict(list)
for log in logs:
    logs_by_task[log["task_id"]].append(log)

tasks_with_multiple_tools = sum(1 for task_logs in logs_by_task.values() if len(task_logs) >= 2)
percent_multiple_tools = (tasks_with_multiple_tools / total_tasks) * 100


# Ручні метрики
manual_metrics = {
    "tasks_with_useful_tool_use": 3,  # Кейси 01, 02, 08 (де номер замовлення справді потрібен)
    "unnecessary_tool_call_count": 5, # Кейси 03, 05, 06, 07, 10 (де номер замовлення або не потрібен, або агенту варто було зрозуміти це з категорії)
    "correct_answers": 4,             # Кейси 01, 02, 06, 09 (відповідь більш-менш адекватна задачі)
    "partly_correct_answers": 2,      # Кейси 04, 08 (є логіка, але є проблеми)
    "wrong_answers": 4,               # Кейси 03, 05, 07, 10 (просить номер замовлення для зміни пароля, при подяці або при вильоті додатку)
    "tasks_tool_ignored": 0,          # Модель ніде не проігнорувала tools (навпаки, аж занадто прислухалась)
    "tasks_contradicts_tool": 0       # Суперечностей немає
}

# Розрахунок відсотків для ручних метрик
percent_ignored = (manual_metrics["tasks_tool_ignored"] / total_tasks) * 100
percent_contradicts = (manual_metrics["tasks_contradicts_tool"] / total_tasks) * 100

print("Метрики")
print(f"1. Tool call success rate: {success_rate:.1f}%")
print(f"2. Average tool calls per task: {calls_per_task:.1f}")
print(f"● Tool error rate: {error_rate:.1f}%")
print(f"● % задач, де agent використав обидва tools: {percent_multiple_tools:.1f}%")
print(f"3. Tasks with useful tool use: {manual_metrics['tasks_with_useful_tool_use']}")
print(f"4. Unnecessary tool call count: {manual_metrics['unnecessary_tool_call_count']}")
print("5. Final answer correctness:")
print(f"   - correct: {manual_metrics['correct_answers']}")
print(f"   - partly correct: {manual_metrics['partly_correct_answers']}")
print(f"   - wrong: {manual_metrics['wrong_answers']}")
print(f"● % задач, де tool output був ignored: {percent_ignored:.1f}%")
print(f"● % задач, де final answer суперечив tool output: {percent_contradicts:.1f}%")

Метрики
1. Tool call success rate: 100.0%
2. Average tool calls per task: 3.0
● Tool error rate: 0.0%
● % задач, де agent використав обидва tools: 100.0%
3. Tasks with useful tool use: 3
4. Unnecessary tool call count: 5
5. Final answer correctness:
   - correct: 4
   - partly correct: 2
   - wrong: 4
● % задач, де tool output був ignored: 0.0%
● % задач, де final answer суперечив tool output: 0.0%


#11. Error analysis

Аналіз 10 показових та проблемних прикладів. Головна виявлена проблема системи — жорстка прив'язка до валідатора, через що агент вимагав номер замовлення навіть там, де це не мало сенсу.

**1. Task ID: case_01**
* **Input:** "Я оплатив замовлення №123456, але статус не змінився."
* **Expected behavior:** Класифікація як `billing` та витяг номера 123456.
* **Actual tool calls:** `classify_ticket` (повернув `general_inquiry`), `extract_entities`, `validate_required_fields`.
* **Final answer:** Агент пообіцяв перевірити статус замовлення №123456.
* **Error category:** `tool output incorrect` (класифікатор не спрацював).
* **Possible fix:** Змінити rule-based класифікатор на такий, що використовує лематизацію (щоб розпізнавати "оплатив" як похідне від "оплата").

**2. Task ID: case_02**
* **Input:** "У мене списало гроші двічі, поверніть!"
* **Expected behavior:** Визначити як `billing`, виявити відсутність номера і попросити його.
* **Actual tool calls:** Усі 3 інструменти. Валідатор повернув `is_valid: false`.
* **Final answer:** Агент ввічливо попросив надати номер замовлення.
* **Error category:** Немає помилки (Показовий успішний кейс).
* **Possible fix:** Залишити як є, система відпрацювала ідеально.

**3. Task ID: case_03**
* **Input:** "Добрий день як там пароль змінити??"
* **Expected behavior:** Класифікація `account` і надання інструкції.
* **Actual tool calls:** Усі 3 інструменти.
* **Final answer:** Агент попросив номер замовлення для підтвердження особи.
* **Error category:** `agent over-trusts tool` / `unnecessary tool call`.
* **Possible fix:** Зробити динамічну маршрутизацію: не викликати валідатор номерів замовлень для категорії `account`.

**4. Task ID: case_04**
* **Input:** "Де мої речі?"
* **Expected behavior:** Запит додаткової інформації.
* **Actual tool calls:** Усі 3 інструменти.
* **Final answer:** Прохання надати номер замовлення.
* **Error category:** Немає помилки (частково успішний логічний кейс).
* **Possible fix:** Відпрацювало нормально.

**5. Task ID: case_05**
* **Input:** "Дякую, все запрацювало, ви найкращі!"
* **Expected behavior:** Ввічлива відповідь без зайвих питань.
* **Actual tool calls:** Усі 3 інструменти.
* **Final answer:** Агент подякував, але попросив номер замовлення.
* **Error category:** `unnecessary tool call`.
* **Possible fix:** Додати категорію `gratitude` або `closing` в класифікатор і пропускати екстракцію сутностей для неї.

**6. Task ID: case_06**
* **Input:** "Не можу зайти в акаунт, мій email test@mail.com"
* **Expected behavior:** Підтвердження проблеми з акаунтом за email.
* **Actual tool calls:** Усі 3 інструменти (витягнуто email).
* **Final answer:** Агент прив'язав проблему блокування до відсутності номера замовлення.
* **Error category:** `agent hallucinates beyond tool output`.
* **Possible fix:** Явно вказати в системному промпті: "Для проблем з акаунтом номер замовлення не є обов'язковим".

**7. Task ID: case_07**
* **Input:** "Додаток вилітає при спробі відкрити налаштування."
* **Expected behavior:** Класифікація `technical` та запит технічних деталей.
* **Actual tool calls:** Усі 3 інструменти.
* **Final answer:** Агент припустив, що виліт пов'язаний із замовленням.
* **Error category:** `agent hallucinates beyond tool output` / `unnecessary tool call`.
* **Possible fix:** Не застосовувати `validate_required_fields` (пошук замовлення) для технічних багів.

**8. Task ID: case_08**
* **Input:** "Скільки коштує доставка для замовлення 98765?"
* **Expected behavior:** Уникнення вигадування ціни, фіксація номера.
* **Actual tool calls:** Усі 3 інструменти. Витягнуто номер 98765.
* **Final answer:** Агент відповів, що ціна залежить від внутрішніх систем, не вигадавши цифру (Baseline вигадав 150 грн).
* **Error category:** Немає помилки (Найкращий показовий кейс).
* **Possible fix:** Залишити як є.

**9. Task ID: case_09**
* **Input:** "Order 445566 is delayed, why?"
* **Expected behavior:** Обробка іншої мови та витяг номера.
* **Actual tool calls:** Усі 3 інструменти, номер витягнуто успішно.
* **Final answer:** Модель відповіла українською, визнавши затримку замовлення 445566.
* **Error category:** Немає помилки (успішна обробка ambiguity/мови).
* **Possible fix:** Залишити як є.

**10. Task ID: case_10**
* **Input:** "Я забув логін. Допоможіть."
* **Expected behavior:** Надання інструкції з відновлення логіна.
* **Actual tool calls:** Усі 3 інструменти.
* **Final answer:** Агент попросив номер останнього замовлення.
* **Error category:** `agent over-trusts tool`.
* **Possible fix:** Гнучкіший промпт і логіка виклику: не всі користувачі мають замовлення, дехто тільки-но зареєструвався.

#12. Generate docs/audit_summary_lab12.md

In [16]:
import os

summary_content = """# Audit Summary Lab 12

1. **Use case**: Support assistant (Варіант 1).
2. **Tools**: `classify_ticket`, `extract_entities`, `validate_required_fields`.
3. **Test cases**: 10.
4. **Tool call success rate**: 100.0%.
5. **Average tool calls per task**: 3.0.
6. **Корисність**: 3 задачі реально виграли від tools (система не вигадувала ціни чи статуси, як це робив baseline).
7. **Зайві виклики**: 5 unnecessary tool calls (агент викликав валідацію номера замовлення навіть для подяк чи технічних багів).
8. **Найкращі приклади**:
   - `case_08`: Агент не став вигадувати ціну доставки (baseline вигадав 150 грн), а спирався на дані систем.
   - `case_02`: Ідеальне відпрацювання браку даних — агент ввічливо попросив номер.
9. **Проблемні приклади**:
   - `case_03`: Агент просив номер замовлення для зміни пароля.
   - `case_05`: Агент вимагав номер замовлення у відповідь на слова "Дякую, все запрацювало".
10. **Що б ви покращували далі**:
   - Реалізувати динамічну маршрутизацію: викликати інструмент `validate_required_fields` тільки якщо класифікатор визначив категорію `billing`.
   - Замінити жорсткий rule-based підхід у `classify_ticket` на семантичний пошук або small LLM, оскільки зараз він не розпізнає похідні слова (наприклад, "оплатив" замість "оплата").
"""

os.makedirs("docs", exist_ok=True)
with open("docs/audit_summary_lab12.md", "w", encoding="utf-8") as f:
    f.write(summary_content)

print("Файл docs/audit_summary_lab12.md згенеровано.")

Файл docs/audit_summary_lab12.md згенеровано.
